In [9]:
import pandas as pd
import numpy as np
import pickle

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report


In [2]:
df = pd.read_csv("/kaggle/input/credit-data1/credit_data.csv")
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month,RISK_SCORE
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,689,0,0,0,0,1,NaN
1,2,120000,2,2,2,26,-1,2,0,0,...,3455,3261,0,1000,1000,1000,0,2000,1,NaN
2,3,90000,2,2,2,34,0,0,0,0,...,14948,15549,1518,1500,1000,1000,1000,5000,0,NaN
3,4,50000,2,2,1,37,0,0,0,0,...,28959,29547,2000,2019,1200,1100,1069,1000,0,NaN
4,5,50000,1,2,1,57,-1,0,-1,0,...,19146,19131,2000,36681,10000,9000,689,679,0,NaN


In [10]:
df.drop(["ID", "RISK_SCORE"], axis=1, errors="ignore", inplace=True)


In [11]:
df.isnull().sum()


LIMIT_BAL                     0
SEX                           0
EDUCATION                     0
MARRIAGE                      0
AGE                           0
PAY_0                         0
PAY_2                         0
PAY_3                         0
PAY_4                         0
PAY_5                         0
PAY_6                         0
BILL_AMT1                     0
BILL_AMT2                     0
BILL_AMT3                     0
BILL_AMT4                     0
BILL_AMT5                     0
BILL_AMT6                     0
PAY_AMT1                      0
PAY_AMT2                      0
PAY_AMT3                      0
PAY_AMT4                      0
PAY_AMT5                      0
PAY_AMT6                      0
default.payment.next.month    0
avg_bill                      0
utilization                   0
delinquency                   0
credit_history                0
avg_payment                   0
dti                           0
limit_ratio                   0
dtype: i

In [14]:
# -------------------------
# UTILIZATION
# -------------------------
bill_cols = [
    "BILL_AMT1","BILL_AMT2","BILL_AMT3",
    "BILL_AMT4","BILL_AMT5","BILL_AMT6"
]

df["avg_bill"] = df[bill_cols].mean(axis=1)
df["utilization"] = df["avg_bill"] / (df["LIMIT_BAL"] + 1)


# -------------------------
# DELINQUENCY
# -------------------------
pay_cols = ["PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6"]
df["delinquency"] = df[pay_cols].clip(lower=0).max(axis=1)


# -------------------------
# CREDIT HISTORY
# -------------------------
df["credit_history"] = (df[pay_cols] > 0).sum(axis=1)


# -------------------------
# DTI (PROXY)
# -------------------------
pay_amt_cols = [
    "PAY_AMT1","PAY_AMT2","PAY_AMT3",
    "PAY_AMT4","PAY_AMT5","PAY_AMT6"
]

df["avg_payment"] = df[pay_amt_cols].mean(axis=1)
df["dti"] = df["avg_payment"] / (df["LIMIT_BAL"] + 1)


# -------------------------
# LIMIT RATIO
# -------------------------
df["limit_ratio"] = df["avg_bill"] / (df["LIMIT_BAL"] + 1)


In [15]:
features = [
    "dti",
    "utilization",
    "limit_ratio",
    "delinquency",
    "credit_history"
]

X = df[features]
y = df["default.payment.next.month"]


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [24]:
model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100
)

model.fit(X_train, y_train)


0:	total: 7.3ms	remaining: 2.18s
100:	total: 538ms	remaining: 1.06s
200:	total: 1.11s	remaining: 549ms
299:	total: 1.67s	remaining: 0us


In [25]:
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))


ROC-AUC: 0.7541752868173901
              precision    recall  f1-score   support

           0       0.83      0.95      0.88      4673
           1       0.62      0.31      0.41      1327

    accuracy                           0.80      6000
   macro avg       0.72      0.63      0.65      6000
weighted avg       0.78      0.80      0.78      6000



In [26]:
feature_importance = model.get_feature_importance(prettified=True)
feature_importance


,Feature Id,Importances
0,delinquency,25.907740
1,credit_history,22.308370
2,dti,20.065502
3,utilization,17.600659
4,limit_ratio,14.117728


In [27]:
new_customer = pd.DataFrame([{
    "dti": 0.75,
    "utilization": 0.90,
    "limit_ratio": 0.85,
    "delinquency": 3,
    "credit_history": 4
}])

model.predict_proba(new_customer)


array([[0.56599997, 0.43400003]])

In [29]:
import numpy as np

def explain_with_shap(model, input_df, feature_names, top_k=3):
    """
    Returns top contributing risk factors for one customer
    """

    # CatBoost native SHAP
    shap_values = model.get_feature_importance(
        data=input_df,
        type="ShapValues"
    )

    # remove base value (last column)
    shap_contrib = shap_values[0][:-1]

    shap_dict = dict(zip(feature_names, shap_contrib))

    # only positive contributors (increase risk)
    positive_risk = {
        k: v for k, v in shap_dict.items() if v > 0
    }

    # sort by impact
    sorted_features = sorted(
        positive_risk.items(),
        key=lambda x: x[1],
        reverse=True
    )

    top_features = [f[0] for f in sorted_features[:top_k]]

    return top_features


In [ ]:
def predict_credit_risk(model, input_df):
    feature_names = input_df.columns.tolist()

    # ---- probability ----
    prob = model.predict_proba(input_df)[0][1]

    # ---- risk level ----
    if prob < 0.3:
        risk_level = "Low"
    elif prob < 0.6:
        risk_level = "Medium"
    else:
        risk_level = "High"

    # ---- SHAP explanation ----
    top_factors = explain_with_shap(
        model,
        input_df,
        feature_names,
        top_k=3
    )

    return {
        "default_probability": round(float(prob), 4),
        "risk_level": risk_level,
        "top_risk_factors": top_factors
    }


In [39]:
from catboost import Pool

# --------------------------------
# create customer input
# --------------------------------
new_customer = pd.DataFrame([{
    "dti": 0.3,
    "utilization": 0.3,
    "limit_ratio": 0.5,
    "delinquency": 1,
    "credit_history": 2
}])

# --------------------------------
# VERY IMPORTANT
# same column order as training
# --------------------------------
new_customer = new_customer[X_train.columns]

# --------------------------------
# convert to Pool
# --------------------------------
customer_pool = Pool(new_customer)

# --------------------------------
# SHAP VALUES
# --------------------------------
shap_values = model.get_feature_importance(
    customer_pool,
    type="ShapValues"
)

# --------------------------------
# remove base value
# --------------------------------
shap_contrib = shap_values[0][:-1]

# --------------------------------
# top positive contributors
# --------------------------------
feature_names = X_train.columns.tolist()

shap_dict = dict(zip(feature_names, shap_contrib))

top_risk_factors = sorted(
    shap_dict.items(),
    key=lambda x: x[1],
    reverse=True
)

top_risk_factors = [k for k, v in top_risk_factors if v > 0][:3]

# --------------------------------
# probability
# --------------------------------
default_prob = model.predict_proba(new_customer)[0][1]

# --------------------------------
# risk level
# --------------------------------
if default_prob < 0.3:
    risk_level = "Low"
elif default_prob < 0.6:
    risk_level = "Medium"
else:
    risk_level = "High"

# --------------------------------
# FINAL OUTPUT
# --------------------------------
ml_output = {
    "default_probability": round(float(default_prob), 4),
    "risk_level": risk_level,
    "top_risk_factors": top_risk_factors
}

ml_output


{'default_probability': 0.1245,
 'risk_level': 'Low',
 'top_risk_factors': ['credit_history']}

In [ ]:
# ============================================
# DUMP MODEL FOR PRODUCTION USE
# ============================================

# Save the CatBoost model in native format
model.save_model('catboost_credit_model.cbm')

print("✓ Model saved: catboost_credit_model.cbm")

# Also verify it works
test_model = CatBoostClassifier()
test_model.load_model('catboost_credit_model.cbm')
print("✓ Model verified - ready for download")

# Download link (Kaggle)
from IPython.display import FileLink
FileLink('catboost_credit_model.cbm')